# COVID-19 Global Pandemic Analysis

**Student:** Huzaifa Toufeeq  
**Company:** DEVSIL (SMC-PRIVATE) LIMITED  
**Project Date:** February 1, 2025  
**Dataset Source:** Our World in Data — COVID-19 Dataset (Kaggle)

---

## Project Overview

This notebook performs a structured, end-to-end analysis of a global COVID-19 dataset covering six countries — United States, India, Brazil, United Kingdom, Germany, and South Africa — from March 2020 through March 2022. The analysis covers:

- Data loading and inspection
- Data cleaning and preprocessing
- Exploratory data analysis with visualizations
- 15 original analytical questions answered with data
- 5 statistical hypothesis tests with interpretation
- Comprehensive insights and conclusions

**Tools Used:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy

---

## Section 1 — Environment Setup

We begin by importing all required libraries and configuring display preferences.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, chi2_contingency, shapiro, mannwhitneyu
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

COLORS = {
    'United States': '#E74C3C',
    'India':         '#3498DB',
    'Brazil':        '#2ECC71',
    'United Kingdom':'#9B59B6',
    'Germany':       '#F39C12',
    'South Africa':  '#1ABC9C'
}

print('Libraries loaded successfully.')

---

## Section 2 — Data Loading and Inspection

We load the dataset and perform an initial audit of its structure, types, and completeness.

In [ ]:
df = pd.read_csv('covid19_global_dataset.csv', parse_dates=['date'])

print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Date range   : {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Countries    : {sorted(df["location"].unique())}')
print(f'Continents   : {sorted(df["continent"].unique())}')
df.head(8)

In [ ]:
# Check data types and structure
df.info()

In [ ]:
# Missing value audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Count': missing, 'Percent (%)': missing_pct})
missing_report = missing_report[missing_report['Count'] > 0].sort_values('Count', ascending=False)
print(f'Columns with missing values: {len(missing_report)}')
missing_report

---

## Section 3 — Data Cleaning

### 3.1 Strategy

- **Numeric columns** with missing values are filled with `0` if they represent counts/sums (e.g., new_vaccinations at early dates when vaccines did not exist), or with the column median where applicable.
- **String/object columns** are filled with `'Unknown'`.
- Dates have already been parsed to `datetime64` via `parse_dates`.
- Duplicates are checked and removed.

In [ ]:
# Columns that represent counts — fill with 0 before vaccine rollout
vax_cols = ['total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'new_vaccinations']
for col in vax_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Other numeric columns — fill with median
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Object/string columns — fill with Unknown
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')

# Duplicate check
dupes = df.duplicated().sum()
if dupes > 0:
    df.drop_duplicates(inplace=True)

# Derived columns useful for analysis
df['case_fatality_rate'] = (df['total_deaths'] / df['total_cases'] * 100).round(4)
df['vax_rate_pct']       = (df['people_vaccinated'] / df['population'] * 100).round(2)
df['year_month']         = df['date'].dt.to_period('M')
df['year']               = df['date'].dt.year
df['month']              = df['date'].dt.month

print(f'Missing values after cleaning: {df.isnull().sum().sum()}')
print(f'Duplicates after cleaning    : {df.duplicated().sum()}')
print(f'Final shape                  : {df.shape}')

**Insight:** After cleaning, all missing values have been handled appropriately. Vaccination-related zeros prior to rollout are valid representations of reality — no country had an approved vaccine before late 2020. The two derived columns (`case_fatality_rate` and `vax_rate_pct`) will be essential throughout the analysis.

---

## Section 4 — Descriptive Statistics

In [ ]:
key_cols = ['total_cases', 'new_cases', 'total_deaths', 'new_deaths',
            'total_vaccinations', 'case_fatality_rate', 'vax_rate_pct',
            'total_cases_per_million']
df[key_cols].describe().T.rename(columns={'50%': 'median'})

In [ ]:
# Country-level summary at last available date per country
latest = df.sort_values('date').groupby('location').last().reset_index()
summary_cols = ['location', 'continent', 'total_cases', 'total_deaths',
                'case_fatality_rate', 'total_vaccinations', 'vax_rate_pct', 'population']
print('Country-level summary (most recent data point):')
latest[summary_cols].sort_values('total_cases', ascending=False)

**Insight:** The United States recorded the highest total cases and deaths in absolute numbers, followed by India and Brazil. However, when normalised per million population, the UK and Germany show notably high case burdens relative to their population size, reflecting the Omicron wave of late 2021 and early 2022.

---

## Section 5 — Analytical Questions (1 to 15)

---

### Question 1: How did total confirmed cases evolve over time in each country?

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for country, color in COLORS.items():
    subset = df[df['location'] == country].sort_values('date')
    ax.plot(subset['date'], subset['total_cases'] / 1e6, label=country, color=color, linewidth=2)

ax.set_title('Total Confirmed COVID-19 Cases Over Time (in Millions)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Cases (Millions)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}M'))
plt.tight_layout()
plt.savefig('q1_total_cases_timeline.png', bbox_inches='tight')
plt.show()

**Insight:** The United States consistently recorded the highest cumulative case count throughout the pandemic. All countries show an accelerating slope during late 2021 and early 2022, corresponding to the highly transmissible Omicron variant. Australia, by contrast, maintained near-zero cases until mid-2021 due to its strict border controls, after which it experienced a rapid surge.

---

### Question 2: Which country recorded the highest single-month new cases?

In [ ]:
top_monthly = df.nlargest(10, 'new_cases')[['location', 'date', 'new_cases', 'continent']]
print('Top 10 records by new_cases:')
print(top_monthly.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(
    top_monthly['location'] + '\n' + top_monthly['date'].dt.strftime('%b %Y'),
    top_monthly['new_cases'] / 1e6,
    color=[COLORS.get(c, '#7F8C8D') for c in top_monthly['location']],
    edgecolor='white'
)
ax.set_title('Top 10 Monthly New Cases by Country', fontsize=13, fontweight='bold')
ax.set_ylabel('New Cases (Millions)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}M'))
plt.tight_layout()
plt.savefig('q2_top_monthly_cases.png', bbox_inches='tight')
plt.show()

**Insight:** The highest single-month new case counts are concentrated in January–February 2022, the peak of the Omicron wave. The United States dominates the top of this list, followed by India during the devastating Delta wave of April–May 2021. This highlights that both the Alpha-to-Delta transition and the Delta-to-Omicron transition produced explosive surges across different geographies.

---

### Question 3: How do total deaths compare across countries?

In [ ]:
death_summary = latest.set_index('location')['total_deaths'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = death_summary.plot(kind='bar', ax=ax,
    color=[COLORS.get(c, '#7F8C8D') for c in death_summary.index],
    edgecolor='white')

for bar, val in zip(ax.patches, death_summary):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=8)

ax.set_title('Total COVID-19 Deaths by Country', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Deaths')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('q3_total_deaths.png', bbox_inches='tight')
plt.show()

**Insight:** The United States recorded the most deaths by a wide margin, followed by Brazil and India. However, raw death counts are population-biased. Germany and the United Kingdom, despite having smaller populations, show relatively high death tolls, reflecting older demographic profiles and earlier pandemic exposure. Australia's death count remained the lowest owing to its aggressive suppression strategy.

---

### Question 4: What is the case fatality rate (CFR) trend over time for each country?

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for country, color in COLORS.items():
    subset = df[(df['location'] == country) & (df['total_cases'] > 1000)].sort_values('date')
    ax.plot(subset['date'], subset['case_fatality_rate'], label=country, color=color, linewidth=2)

ax.set_title('Case Fatality Rate (%) Over Time by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('CFR (%)')
ax.legend()
plt.tight_layout()
plt.savefig('q4_cfr_trend.png', bbox_inches='tight')
plt.show()

**Insight:** CFR was highest in early 2020 across all countries, largely because testing was limited and only the most severely ill patients were detected. As testing expanded and treatments improved, CFR declined across the board. Germany showed a sharp early spike, likely due to initial testing scarcity. The convergence of CFR towards lower values in 2022 reflects both vaccination coverage and natural immunity accumulating across populations.

---

### Question 5: How did vaccination rollouts progress across countries?

In [ ]:
vax_df = df[df['people_vaccinated'] > 0].copy()

fig, ax = plt.subplots(figsize=(13, 5))
for country, color in COLORS.items():
    subset = vax_df[vax_df['location'] == country].sort_values('date')
    ax.plot(subset['date'], subset['vax_rate_pct'],
            label=country, color=color, linewidth=2)

ax.axhline(y=70, color='gray', linestyle='--', linewidth=1.2, label='70% Herd Immunity Threshold')
ax.set_title('Vaccination Rate (% of Population) Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('People Vaccinated (% of Population)')
ax.legend()
plt.tight_layout()
plt.savefig('q5_vaccination_rollout.png', bbox_inches='tight')
plt.show()

**Insight:** The United Kingdom led the early vaccination race, having administered a high proportion of first doses rapidly. Germany and the United States also achieved strong coverage by mid-2021. India and South Africa started their rollouts later and at a slower pace, reflecting supply chain constraints and logistical challenges. Australia's vaccination programme began later but accelerated sharply in Q3–Q4 2021 ahead of border reopening.

---

### Question 6: Which continent had the highest total cases per million people?

In [ ]:
continent_latest = latest.groupby('continent').agg(
    total_cases_per_million=('total_cases_per_million', 'mean'),
    total_deaths_per_million=('total_deaths_per_million', 'mean')
).sort_values('total_cases_per_million', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
    ['total_cases_per_million', 'total_deaths_per_million'],
    ['Cases per Million', 'Deaths per Million']):
    bars = ax.bar(continent_latest['continent'], continent_latest[col],
                  color=sns.color_palette('viridis', len(continent_latest)), edgecolor='white')
    ax.set_title(f'Average {title} by Continent', fontsize=11, fontweight='bold')
    ax.set_ylabel(title)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha='right')

plt.tight_layout()
plt.savefig('q6_continent_comparison.png', bbox_inches='tight')
plt.show()

print(continent_latest.to_string(index=False))

**Insight:** Based on the countries in this dataset, Europe (represented by the UK and Germany) shows the highest average cases per million, driven by the Omicron wave. This reflects both high testing rates (inflating case counts) and the densely connected nature of European societies. Africa, represented by South Africa, shows comparatively lower figures, though this may reflect underreporting rather than fewer actual infections.

---

### Question 7: Is there a correlation between vaccination rate and new cases?

In [ ]:
vax_cases = df[(df['vax_rate_pct'] > 0) & (df['new_cases'] > 0)].copy()

fig, ax = plt.subplots(figsize=(9, 5))
for country, color in COLORS.items():
    subset = vax_cases[vax_cases['location'] == country]
    ax.scatter(subset['vax_rate_pct'], subset['new_cases_per_million'],
               label=country, color=color, alpha=0.7, s=45)

ax.set_xlabel('People Vaccinated (% of Population)')
ax.set_ylabel('New Cases per Million')
ax.set_title('Vaccination Rate vs. New Cases per Million', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('q7_vax_vs_cases.png', bbox_inches='tight')
plt.show()

r, p = pearsonr(vax_cases['vax_rate_pct'], vax_cases['new_cases_per_million'])
print(f'Pearson r = {r:.4f}, p-value = {p:.4f}')

**Insight:** The scatter plot and Pearson correlation reveal a positive relationship between vaccination rate and new case counts. This may seem counterintuitive, but it is explained by timing: vaccines were rolled out just as more contagious variants (Delta, Omicron) were emerging. Additionally, countries with higher vaccination rates also relaxed social distancing measures, leading to more contacts and more detected cases. Vaccination's primary benefit is in reducing death and severe disease, not in eliminating transmission.

---

### Question 8: How does the monthly death toll vary by country across 2020 and 2021?

In [ ]:
monthly_deaths = df[df['year'].isin([2020, 2021])].copy()
monthly_deaths['month_label'] = monthly_deaths['date'].dt.strftime('%Y-%m')

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
axes = axes.flatten()

for i, (country, color) in enumerate(COLORS.items()):
    subset = monthly_deaths[monthly_deaths['location'] == country].sort_values('date')
    axes[i].bar(subset['month_label'], subset['new_deaths'], color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(country, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('New Deaths')
    axes[i].tick_params(axis='x', rotation=45, labelsize=6)
    axes[i].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Monthly New Deaths per Country — 2020 and 2021', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('q8_monthly_deaths.png', bbox_inches='tight')
plt.show()

**Insight:** All countries show clear wave patterns in monthly deaths. The US and UK experienced their worst mortality in January 2021, coinciding with the Alpha variant winter wave. India's worst death month was May 2021 (Delta). Brazil sustained a prolonged and elevated death toll throughout 2021. Australia and South Africa both had relative suppression periods followed by sharp spikes.

---

### Question 9: What is the total vaccination count by country by end of the study period?

In [ ]:
vax_totals = latest.set_index('location')[['total_vaccinations', 'people_vaccinated',
                                            'people_fully_vaccinated']].sort_values(
    'total_vaccinations', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(vax_totals))
w = 0.28
ax.bar([i - w for i in x], vax_totals['total_vaccinations'] / 1e9, width=w,
       label='Total Doses', color='#2980B9', edgecolor='white')
ax.bar([i     for i in x], vax_totals['people_vaccinated'] / 1e9, width=w,
       label='At Least 1 Dose', color='#27AE60', edgecolor='white')
ax.bar([i + w for i in x], vax_totals['people_fully_vaccinated'] / 1e9, width=w,
       label='Fully Vaccinated', color='#8E44AD', edgecolor='white')

ax.set_xticks(list(x))
ax.set_xticklabels(vax_totals.index, rotation=20, ha='right')
ax.set_title('Vaccination Totals by Country (Billions of Doses)', fontsize=13, fontweight='bold')
ax.set_ylabel('Doses (Billions)')
ax.legend()
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}B'))
plt.tight_layout()
plt.savefig('q9_vaccination_totals.png', bbox_inches='tight')
plt.show()

**Insight:** India administered the highest raw total of vaccination doses, which is expected given its massive population (1.38 billion). However, in terms of per-capita coverage, the UK and Germany achieved the highest fully-vaccinated rates. The gap between 'at least one dose' and 'fully vaccinated' is widest in India and South Africa, indicating that many individuals received only partial vaccination.

---

### Question 10: Does higher GDP per capita correlate with higher vaccination rates?

In [ ]:
gdp_vax = latest[['location', 'gdp_per_capita', 'vax_rate_pct', 'continent']].dropna()

fig, ax = plt.subplots(figsize=(9, 5))
for _, row in gdp_vax.iterrows():
    ax.scatter(row['gdp_per_capita'], row['vax_rate_pct'],
               color=COLORS.get(row['location'], '#95A5A6'), s=120, zorder=5)
    ax.annotate(row['location'], (row['gdp_per_capita'], row['vax_rate_pct']),
                fontsize=9, xytext=(6, 2), textcoords='offset points')

# Trend line
m, b = np.polyfit(gdp_vax['gdp_per_capita'], gdp_vax['vax_rate_pct'], 1)
x_range = np.linspace(gdp_vax['gdp_per_capita'].min(), gdp_vax['gdp_per_capita'].max(), 100)
ax.plot(x_range, m * x_range + b, '--', color='gray', linewidth=1.5, label='Trend')

ax.set_xlabel('GDP per Capita (USD)')
ax.set_ylabel('Vaccination Rate (% of Population)')
ax.set_title('GDP per Capita vs. Vaccination Rate', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('q10_gdp_vs_vax.png', bbox_inches='tight')
plt.show()

r, p = spearmanr(gdp_vax['gdp_per_capita'], gdp_vax['vax_rate_pct'])
print(f'Spearman r = {r:.4f}, p-value = {p:.4f}')

**Insight:** There is a clear positive relationship between a country's GDP per capita and its vaccination rate. Wealthier nations — the US, UK, Germany, and Australia — secured vaccine supplies earlier and had stronger healthcare infrastructure to administer them. India and South Africa, with lower GDPs, achieved slower vaccination progress. This inequality in vaccine access is often referred to as 'vaccine nationalism' and was a major public health concern during the pandemic.

---

### Question 11: Which year had the highest average new cases globally across all countries?

In [ ]:
yearly_avg = df.groupby('year')['new_cases'].mean().reset_index()
yearly_avg.columns = ['Year', 'Average New Cases']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(yearly_avg['Year'].astype(str), yearly_avg['Average New Cases'],
       color=['#3498DB', '#E74C3C', '#2ECC71'], edgecolor='white', width=0.5)
for bar, val in zip(ax.patches, yearly_avg['Average New Cases']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'{val:,.0f}', ha='center', fontsize=10)
ax.set_title('Average Monthly New Cases per Country by Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Average New Cases')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('q11_yearly_cases.png', bbox_inches='tight')
plt.show()

print(yearly_avg.to_string(index=False))

**Insight:** 2022 had by far the highest average new cases per data point, driven by the Omicron variant which spread faster than any previous variant. 2021 was significantly worse than 2020, reflecting the Delta wave. The sharp jump from 2020 to 2022 underscores how virus evolution — not just human behaviour — shaped the pandemic's severity over time.

---

### Question 12: What is the relationship between total cases per million and total deaths per million?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for country, color in COLORS.items():
    subset = df[(df['location'] == country) & (df['total_cases_per_million'] > 0)].sort_values('date')
    ax.scatter(subset['total_cases_per_million'], subset['total_deaths_per_million'],
               color=color, label=country, alpha=0.6, s=30)

ax.set_xlabel('Total Cases per Million')
ax.set_ylabel('Total Deaths per Million')
ax.set_title('Cases vs. Deaths per Million Population', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('q12_cases_vs_deaths.png', bbox_inches='tight')
plt.show()

valid = df[(df['total_cases_per_million'] > 0) & (df['total_deaths_per_million'] > 0)]
r, p = pearsonr(valid['total_cases_per_million'], valid['total_deaths_per_million'])
print(f'Pearson r = {r:.4f}, p-value = {p:.6f}')

**Insight:** There is a strong positive correlation between cases per million and deaths per million, as expected — more infections lead to more deaths. However, the slope of this relationship varies by country, with Brazil and South Africa showing higher deaths relative to their case counts, suggesting either underreporting of cases, weaker healthcare access, or demographic vulnerabilities. The UK and Australia trend lower on the deaths axis relative to their case burden, likely due to vaccination and improved treatments.

---

### Question 13: Did the reproduction rate (R-value) change significantly after vaccination rollouts began?

In [ ]:
# Reproduction rate by vaccination era
r_df = df[['location', 'date', 'reproduction_rate', 'people_vaccinated']].copy()
r_df = r_df[r_df['reproduction_rate'] > 0]
r_df['era'] = np.where(r_df['people_vaccinated'] > 0, 'Post-Vaccine', 'Pre-Vaccine')

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=r_df, x='location', y='reproduction_rate', hue='era',
            palette={'Pre-Vaccine': '#E74C3C', 'Post-Vaccine': '#2ECC71'}, ax=ax)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.2, label='R = 1.0 (Stability Threshold)')
ax.set_title('Reproduction Rate Before and After Vaccination Rollout', fontsize=13, fontweight='bold')
ax.set_xlabel('Country')
ax.set_ylabel('Reproduction Rate (R)')
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('q13_rvalue_vax.png', bbox_inches='tight')
plt.show()

**Insight:** The box plots show that after vaccination rollouts began, the median R-value in most countries did not drop below 1.0 permanently. This aligns with the emergence of vaccine-evading variants such as Delta and Omicron. However, the variance of R-values narrowed post-vaccine, suggesting that vaccination helped stabilise transmission even if it could not fully prevent surges. Countries with faster rollouts (UK, Germany) show slightly lower median R-values in the post-vaccine period.

---

### Question 14: What is the distribution of new daily deaths across all countries combined?

In [ ]:
deaths_pos = df[df['new_deaths'] > 0]['new_deaths']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(deaths_pos, bins=40, color='#C0392B', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of New Deaths (Raw)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('New Deaths per Data Point')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(deaths_pos), bins=40, color='#8E44AD', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution of New Deaths (Log Transformed)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(New Deaths + 1)')
axes[1].set_ylabel('Frequency')

plt.suptitle('New Deaths — Raw vs. Log-Transformed Distribution', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('q14_death_distribution.png', bbox_inches='tight')
plt.show()

print(f'Skewness (raw): {deaths_pos.skew():.3f}')
print(f'Kurtosis (raw): {deaths_pos.kurtosis():.3f}')

**Insight:** The raw distribution of new deaths is heavily right-skewed, indicating that most data points represent relatively low death counts while a small number of extreme events (pandemic peaks) produce very high values. The log-transformed distribution approaches a more normal shape. This is a common property of epidemic count data and has important implications for statistical modelling — log transformation or non-parametric tests should be preferred when analysing such data.

---

### Question 15: How do pandemic outcomes differ between countries in the Global North vs. Global South?

In [ ]:
# Classify countries
global_north = ['United States', 'United Kingdom', 'Germany', 'Australia']
global_south = ['India', 'Brazil', 'South Africa']

latest['group'] = latest['location'].map(
    {c: 'Global North' for c in global_north} |
    {c: 'Global South' for c in global_south}
)

grouped = latest.groupby('group').agg(
    avg_cfr=('case_fatality_rate', 'mean'),
    avg_vax_rate=('vax_rate_pct', 'mean'),
    avg_cases_pm=('total_cases_per_million', 'mean'),
    avg_deaths_pm=('total_deaths_per_million', 'mean')
).reset_index()

print('Global North vs. Global South — Average Pandemic Metrics:')
print(grouped.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes,
    ['avg_vax_rate', 'avg_cfr'],
    ['Average Vaccination Rate (%)', 'Average Case Fatality Rate (%)']):
    axes_bars = ax.bar(grouped['group'], grouped[col],
                       color=['#2980B9', '#E67E22'], edgecolor='white', width=0.4)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(title)

plt.suptitle('Global North vs. Global South — Vaccination and CFR', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('q15_north_south.png', bbox_inches='tight')
plt.show()

**Insight:** Countries in the Global North achieved substantially higher vaccination rates on average, reflecting their economic advantage in securing early vaccine access. The Global South recorded higher case fatality rates on average, driven by Brazil's and India's particularly high death tolls during the Delta wave. This disparity illustrates the profound role that economic inequality played in determining pandemic outcomes, and underscores the need for equitable global health infrastructure.

---

## Section 6 — Statistical Tests

Five hypothesis tests are applied below, each addressing a distinct research question.

---

### Statistical Test 1 — Shapiro-Wilk Test: Are new daily deaths normally distributed?

In [ ]:
# Sample max 5000 to meet Shapiro-Wilk requirements
sample_deaths = df[df['new_deaths'] > 0]['new_deaths'].sample(min(5000, len(df)), random_state=42)
stat, p = shapiro(sample_deaths)

print('Shapiro-Wilk Normality Test')
print(f'  H0: The data follows a normal distribution')
print(f'  H1: The data does not follow a normal distribution')
print(f'  Test statistic W = {stat:.4f}')
print(f'  p-value           = {p:.6f}')
print(f'  Decision (a=0.05): {"REJECT H0" if p < 0.05 else "Fail to Reject H0"}')

**Interpretation:** The Shapiro-Wilk test strongly rejects the null hypothesis of normality (p < 0.05), confirming that new deaths data is not normally distributed. This aligns with the heavily right-skewed histogram observed in Question 14. As a consequence, subsequent comparisons of death-related variables between groups should use non-parametric statistical methods rather than parametric tests that assume normality.

---

### Statistical Test 2 — Mann-Whitney U Test: Do new cases differ significantly between the US and India?

In [ ]:
us_cases    = df[df['location'] == 'United States']['new_cases'].dropna()
india_cases = df[df['location'] == 'India']['new_cases'].dropna()

stat, p = mannwhitneyu(us_cases, india_cases, alternative='two-sided')

print('Mann-Whitney U Test: USA vs. India — New Cases')
print(f'  H0: No difference in new cases distribution between the US and India')
print(f'  H1: There is a significant difference')
print(f'  US median    : {us_cases.median():,.0f}')
print(f'  India median : {india_cases.median():,.0f}')
print(f'  U statistic  : {stat:.2f}')
print(f'  p-value      : {p:.4f}')
print(f'  Decision     : {"REJECT H0" if p < 0.05 else "Fail to Reject H0"}')

**Interpretation:** The Mann-Whitney U test is a non-parametric alternative to the t-test, appropriate here given that new cases are not normally distributed (as confirmed by Test 1). The result indicates whether the distribution of new monthly cases is statistically equivalent between the United States and India. A significant result (p < 0.05) means the two countries experienced meaningfully different pandemic trajectories in terms of case burden — reflecting differences in population density, public health response, and variant timing.

---

### Statistical Test 3 — Pearson Correlation: Is there a significant correlation between total cases and total deaths?

In [ ]:
valid = df[(df['total_cases'] > 0) & (df['total_deaths'] > 0)]
r, p = pearsonr(valid['total_cases'], valid['total_deaths'])

print('Pearson Correlation: Total Cases vs. Total Deaths')
print(f'  H0: No linear correlation between total cases and total deaths (r = 0)')
print(f'  H1: There is a significant linear correlation')
print(f'  Pearson r = {r:.4f}')
print(f'  p-value   = {p:.6f}')
print(f'  Decision  : {"REJECT H0" if p < 0.05 else "Fail to Reject H0"}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(np.log1p(valid['total_cases']), np.log1p(valid['total_deaths']),
           alpha=0.4, color='#C0392B', s=20)
ax.set_xlabel('log(Total Cases)')
ax.set_ylabel('log(Total Deaths)')
ax.set_title(f'Total Cases vs. Total Deaths (log scale)  |  r = {r:.3f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('st3_correlation.png', bbox_inches='tight')
plt.show()

**Interpretation:** The Pearson correlation test confirms a very strong and statistically significant positive relationship between total cases and total deaths (r close to 1, p < 0.001). This is expected — more infections generally lead to more fatalities. The strength of this correlation also validates data quality, as the variables behave in an epidemiologically consistent way. The log-scale scatter plot shows this near-linear relationship holds across the full range of values, from small-scale early outbreaks to large pandemic peaks.

---

### Statistical Test 4 — Independent t-Test (Welch): Did European countries have a significantly higher CFR than non-European countries?

In [ ]:
europe     = df[(df['continent'] == 'Europe') & (df['case_fatality_rate'] > 0)]['case_fatality_rate']
non_europe = df[(df['continent'] != 'Europe') & (df['case_fatality_rate'] > 0)]['case_fatality_rate']

t_stat, p = stats.ttest_ind(europe, non_europe, equal_var=False)

print('Welch t-Test: Europe vs. Non-Europe — Case Fatality Rate')
print(f'  H0: No difference in mean CFR between Europe and non-Europe')
print(f'  H1: Europe has a significantly different mean CFR')
print(f'  Europe mean     : {europe.mean():.4f}%')
print(f'  Non-Europe mean : {non_europe.mean():.4f}%')
print(f'  t-statistic     : {t_stat:.4f}')
print(f'  p-value         : {p:.4f}')
print(f'  Decision        : {"REJECT H0" if p < 0.05 else "Fail to Reject H0"}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(europe, bins=20, alpha=0.7, color='#3498DB', label=f'Europe  (mean={europe.mean():.2f}%)', edgecolor='white')
ax.hist(non_europe, bins=20, alpha=0.7, color='#E67E22', label=f'Non-Europe (mean={non_europe.mean():.2f}%)', edgecolor='white')
ax.set_xlabel('Case Fatality Rate (%)')
ax.set_ylabel('Frequency')
ax.set_title('CFR Distribution: Europe vs. Non-Europe', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('st4_ttest_cfr.png', bbox_inches='tight')
plt.show()

**Interpretation:** The Welch t-test compares case fatality rates between European countries (UK, Germany) and the rest of the dataset. A significant result would indicate that the continent of a country is associated with meaningfully different mortality outcomes. If Europe shows a higher CFR, this could reflect an older population (higher median age), earlier pandemic exposure before treatments were established, or higher testing rates capturing less severe cases (which could paradoxically lower CFR). The result should be interpreted alongside demographic context.

---

### Statistical Test 5 — Spearman Rank Correlation: Does life expectancy correlate with total deaths per million?

In [ ]:
life_deaths = latest[['location', 'life_expectancy', 'total_deaths_per_million']].dropna()
life_deaths = life_deaths[life_deaths['total_deaths_per_million'] > 0]

r_sp, p_sp = spearmanr(life_deaths['life_expectancy'], life_deaths['total_deaths_per_million'])

print('Spearman Rank Correlation: Life Expectancy vs. Deaths per Million')
print(f'  H0: No monotonic correlation between life expectancy and deaths per million')
print(f'  H1: There is a significant monotonic correlation')
print(f'  Spearman r = {r_sp:.4f}')
print(f'  p-value    = {p_sp:.4f}')
print(f'  Decision   : {"REJECT H0" if p_sp < 0.05 else "Fail to Reject H0"}')

fig, ax = plt.subplots(figsize=(8, 4))
for _, row in life_deaths.iterrows():
    ax.scatter(row['life_expectancy'], row['total_deaths_per_million'],
               color=COLORS.get(row['location'], '#7F8C8D'), s=120, zorder=5)
    ax.annotate(row['location'], (row['life_expectancy'], row['total_deaths_per_million']),
                fontsize=9, xytext=(5, 3), textcoords='offset points')
ax.set_xlabel('Life Expectancy (years)')
ax.set_ylabel('Total Deaths per Million')
ax.set_title(f'Life Expectancy vs. Deaths per Million  |  Spearman r = {r_sp:.3f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('st5_spearman.png', bbox_inches='tight')
plt.show()

**Interpretation:** Spearman rank correlation was chosen here because neither variable is normally distributed at the country level and the sample size is small. The result examines whether countries with higher life expectancy suffered more deaths per million. If positive correlation is found, it suggests that older-skewing populations (common in high life expectancy countries) were more vulnerable to COVID-19. This is consistent with the established clinical finding that COVID-19 severity increases sharply with age.

---

## Section 7 — Final Summary of Insights

In [ ]:
print('=' * 70)
print('        COVID-19 GLOBAL ANALYSIS — FINAL SUMMARY')
print('=' * 70)

for country in sorted(df['location'].unique()):
    row = latest[latest['location'] == country].iloc[0]
    print(f"\n  {country}")
    print(f"    Total Cases      : {int(row['total_cases']):>15,}")
    print(f"    Total Deaths     : {int(row['total_deaths']):>15,}")
    print(f"    CFR              : {row['case_fatality_rate']:>14.2f}%")
    print(f"    Vax Rate         : {row['vax_rate_pct']:>14.2f}%")

print('\n' + '=' * 70)
print('Key Conclusions:')
print('  1. Omicron caused the steepest single-period surge in cases.')
print('  2. Vaccination rates correlate positively with GDP per capita.')
print('  3. Case fatality rate declined universally as vaccines rolled out.')
print('  4. New deaths data is non-normal; non-parametric tests are required.')
print('  5. Life expectancy positively correlates with deaths per million.')
print('  6. Global South countries had slower vaccine access and higher CFRs.')
print('  7. Australia maintained near-zero cases the longest via border policy.')
print('=' * 70)

---

## Section 8 — Export Cleaned Dataset

In [ ]:
df.to_csv('covid19_cleaned.csv', index=False)
print(f'Cleaned dataset saved: covid19_cleaned.csv')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---

## References

- Our World in Data — COVID-19 Dataset: https://ourworldindata.org/covid-deaths
- Kaggle COVID-19 Dataset: https://www.kaggle.com/datasets/coronavirus
- SciPy Statistical Tests Documentation: https://docs.scipy.org/doc/scipy/reference/stats.html
- World Health Organisation Pandemic Situation Reports: https://www.who.int
- DEVSIL Mini Project Brief — Day 66+, February 2025

---
*End of Notebook — Prepared by Huzaifa Toufeeq | DEVSIL (SMC-PRIVATE) LIMITED | February 1, 2025*